# Lab 06 · Multi-agent debate: propose, challenge, agree

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Save `FREEINFERENCE_API_KEY`, `DEEPSEEK_API_KEY`, and `GEMINI_API_KEY` in Colab Secrets. Each architect uses its own provider and call budget.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask

## Warm-up · Proposal and challenge

Run this cell with your DeepSeek key. The first model call proposes a rainy-day activity. The second receives that answer and looks for a problem with it. Read the two replies in order to see how one answer becomes input to a critique. The main exercise adds separate reviewers and a vote.

In [ ]:
lunch_ask = make_ask(provider="deepseek")  # DeepSeek is one of the three required providers.
idea = lunch_ask("Propose one outdoor activity for a rainy afternoon. One sentence.")
challenge = lunch_ask(f"Challenge this proposal for rainy weather in one sentence: {idea}")
print("Proposal:", idea)
print("Challenge:", challenge)

**Next:** Now gather independent architecture proposals, challenge unsupported claims, and vote.

## Multi-agent debate: several agents, one decision
ShopEasy support needs to search orders, payments, and deliveries by customer name. It can collect records from each service when someone searches (**API composition**) or search a separate view that is updated as those records change (**CQRS**). A separate view can make broad searches faster, but updates may take time to appear.

An **agent** here is a model reviewer with a role and instructions. Three reviewers receive the same source notes. Each proposes an option without seeing the others' first answers. Python coordinates the reviewers as the **chair**. It checks each proposal, asks for a challenge and revision, and counts the eligible votes.

Follow the **transcript**, the ordered record of proposals, challenges, revisions, and the outcome. Look for a decision when at least two reviewers back the same option, or an escalation when a person needs to decide.

**At a glance**

![Lab 06 at a glance: Shared sources (R2 · R3 · case), then Minimalist (Qwen) or Safeguard (DeepSeek) or Modernizer (Gemini), then valid citations?, then Challenge + revise (one round), then Chair counts (⅔ of registered peers), then Decision (+ dissent kept) or Escalate (no majority).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/06.svg)

## Think about the example

ShopEasy support needs to search orders, payments, and deliveries by customer name. API composition joins responses; CQRS provides a searchable read model that may lag behind writes.

**Concepts:** API composition versus CQRS; Eventual consistency and read-model cost.

Use the scenario above to consider these questions before running the code:

1. Why is a customer-name search different from a lookup by order ID?
2. Which cost of CQRS must the council record?

## The idea in context

Independent proposals do not see one another. A coded chair limits discussion and aggregates proposals; agreement is not proof of correctness.

CQRS supports search but adds a read model and consistency lag; composition has different costs.

| Key term | Meaning |
| --- | --- |
| **Multi-agent system** | Several model-driven agents with separate roles that work on one task. |
| **Independence** | Each agent answers without seeing the others first, so its view is its own. |
| **Grounding** | Supporting a claim with a source that everyone can check. |
| **Escalation** | Handing the decision to a person when the agents cannot agree. |

## Pattern · Debate-Based Cooperation

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Voting-Based Cooperation, Shared Memory.*

| | |
| --- | --- |
| **Context** | Hard questions hold competing values and assumptions that become clearer through structured disagreement. |
| **Problem** | Independent answers never confront one another, so weak claims and hidden trade-offs survive aggregation. |
| **Solution** | Let agents with contrasting positions respond to one another over bounded rounds, using a shared transcript, then ask a judge for a structured verdict. |
| **Use when** | Trade-offs are genuinely contested; reasoning quality matters more than speed; a transcript is useful evidence. |
| **Trade-off** | Debate surfaces assumptions, but it costs many tokens, and persuasive rhetoric can outweigh truth. |
| **Flow** | Positions → Debate K rounds → Judge → Verdict |
| **Quality attributes** | Benefits: Reliability, Accuracy, Auditability. Liabilities: Latency, Cost, Accuracy, Scalability, Predictability. |
| **In this lab** | The proposals are independent, and the transcript is the debate's *Shared Memory*. The chair is a judge with fixed rules: it requests one round of critique for proposals that fail the checks. When all proposals pass, it picks one for a classroom follow-up. It then counts eligible votes as in *Voting-Based Cooperation*. Source labels give you a place to start checking the claims. |

## Main exercise at a glance

1. Shared sources and citation rules
2. Three architects and providers
3. Collect independent proposals
4. Challenge unsupported claims
5. Count valid votes and keep dissent

## Step 1 · Shared sources and citation rules
The cell prints four short source notes with IDs such as `R3` and `C1`. A **citation** names one of those notes. Every reviewer receives the same notes, and every vote needs at least one citation with all cited IDs drawn from this list.

Later, when a reviewer cites `R3`, read that note and compare it with the claim. The code checks that the ID exists. You assess whether the note supports what the reviewer said.

In [ ]:
sources = {
    "R2": "API composition joins small results by id; broad cross-service search is a poor fit.",
    "R3": "CQRS supports rich queries with a separate read model; the view is eventually consistent.",
    "C1": "ShopEasy support must search orders, payments and deliveries by customer name.",
    "C2": "Support accepts a slightly stale view, but the team wants few moving parts.",
}
evidence = ""
for source_id, text in sources.items():
    evidence += f"[{source_id}] {text}\n"
print(evidence)

OPTIONS = ("API composition", "CQRS")

def valid_proposal(message):
    claim, citations = message["claim"], message["evidence"]
    return (message["option"] in OPTIONS and isinstance(claim, str) and bool(claim.strip())
            and isinstance(citations, list) and bool(citations)
            and all(isinstance(source_id, str) and source_id in sources for source_id in citations))

## Step 2 · Three architects and providers
A **stance** gives each model reviewer a priority, such as keeping the design simple or reducing risk. A **provider** hosts a model and gives the course code a way to call it. This exercise uses Qwen through FreeInference, plus DeepSeek and Gemini.

Save `FREEINFERENCE_API_KEY`, `DEEPSEEK_API_KEY`, and `GEMINI_API_KEY` in Colab Secrets and grant this notebook access. The warm-up only uses DeepSeek. For the main exercise, this cell connects one model client per reviewer and prints the three model families. If a key is missing, check its exact name and access before rerunning the cell.

In [ ]:
from coursekit import PROVIDERS

council = {
    "Minimalist": {"provider": "freeinference", "stance": "Prefer the simplest design that meets the requirements."},
    "Safeguard": {"provider": "deepseek", "stance": "Prefer low-risk designs; weigh operations and failures."},
    "Modernizer": {"provider": "gemini", "stance": "Prefer designs that keep search responsive as order history grows."},
}
families = {}
asks = {}
for name, architect in council.items():
    provider = architect["provider"]
    families[name] = PROVIDERS[provider]["family"]
    asks[name] = make_ask(provider=provider)
print("Model families:", families)
assert len(set(families.values())) >= 3, "The council needs at least three model families."

## Step 3 · Collect independent proposals
Python sends one request per reviewer. Each prompt has the reviewer's stance, the same source notes, and the same question. The reviewers see one another's ideas when the chair requests a challenge in Step 4.

Look for three printed replies. Each reply should be a JSON object with an `option` to vote for, a short `claim` explaining why, and an `evidence` list of source IDs. Python records the fields it can read and keeps going if a reply is malformed. The chair checks those fields before counting any vote.

In [ ]:
import json

REPLY = 'Reply with JSON only, no code fences: {"option": "...", "claim": "one sentence", "evidence": ["ids"]}'

def proposal_record(reply, name, round_number, kind):
    try:
        body = json.loads(reply)
    except json.JSONDecodeError:
        body = None
    if not isinstance(body, dict):
        body = {}
    return {"round": round_number, "speaker": name, "kind": kind,
            "option": body.get("option"), "claim": body.get("claim"), "evidence": body.get("evidence")}

proposals = {}
for name, architect in council.items():
    role = f"You are the {name} on an architecture council. {architect['stance']}"
    task = f"Choose API composition or CQRS for ShopEasy support search. {REPLY}"
    reply = asks[name](f"{role}\nSources:\n{evidence}\n{task}")
    print(name, "replied:", reply)
    proposals[name] = proposal_record(reply, name, 1, "proposal")

## Step 4 · Challenge unsupported claims
The chair asks for a challenge if a proposal has an unknown option, a missing claim, or missing or unknown citations. If all three proposals pass the check, it asks the first reviewer to strengthen their proposal as a classroom example. You will see a challenge in either case.

A critic with a different recommendation speaks first if one is available. The author gets one revision, and Python keeps the original proposal in the transcript. A real source ID gives you a note to inspect. Read the claim and note together to decide whether it supports the argument.

In [ ]:
transcript = list(proposals.values())
final_proposals = proposals.copy()
all_valid = all(valid_proposal(message) for message in proposals.values())
follow_up = next(iter(proposals)) if all_valid else None
if follow_up:
    print("Classroom follow-up: all proposals passed, so challenge", follow_up)
for name, message in proposals.items():
    if valid_proposal(message) and name != follow_up:
        continue
    peers = [peer for peer in council if peer != name]
    critic = next((peer for peer in peers if proposals[peer]["option"] in OPTIONS
                   and proposals[peer]["option"] != message["option"]), peers[0])
    role = f"You are the {critic}. {council[critic]['stance']}"
    task = f"In one sentence, challenge this proposal by {name}: {message}"
    challenge = asks[critic](f"{role}\nSources:\n{evidence}\n{task}")
    transcript.append({"round": 2, "speaker": critic, "kind": "challenge", "target": name, "text": challenge})
    role = f"You are the {name}. {council[name]['stance']}"
    task = (f"Your first proposal: {message}\nChallenge: {challenge}\n"
            f"Revise your proposal. Choose API composition or CQRS and cite known source IDs. {REPLY}")
    revision = asks[name](f"{role}\nSources:\n{evidence}\n{task}")
    print(f"{critic} challenges {name}: {challenge}")
    print(f"{name} revision reply:", revision)
    revised = proposal_record(revision, name, 2, "revision")
    final_proposals[name] = revised
    transcript.append(revised)

## Step 5 · Count valid votes and keep dissent
Python counts one vote for each reviewer's latest proposal that names an allowed option, explains the choice, and cites known sources. A decision requires two of the three reviewers to select the same option. If that threshold is missed, the chair records an **escalation** for a person to resolve.

Read the lines marked `proposal`, `challenge`, `revision`, and `decision` or `escalation`. **Dissent** is the argument from a reviewer who preferred a different option. The transcript keeps dissent available to anyone checking the decision.

In [ ]:
from collections import Counter

votes = Counter()
for message in final_proposals.values():
    if valid_proposal(message):
        votes[message["option"]] += 1
option, count = votes.most_common(1)[0] if votes else (None, 0)
decided = count / len(council) >= 2 / 3
dissent = []
for message in final_proposals.values():
    if decided and valid_proposal(message) and message["option"] != option:
        dissent.append(message)
outcome = {"round": 3, "speaker": "Chair", "kind": "decision" if decided else "escalation",
           "option": option if decided else None, "votes": dict(votes), "dissent": dissent}
transcript.append(outcome)
for message in transcript:
    print("round", message["round"], message["speaker"], message["kind"], message.get("text") or message["option"])
print("Votes:", outcome["votes"])
for message in dissent:
    print("Dissent:", message["speaker"], message["claim"])

## What you should see

Read `Votes:` for the totals, then find the chair's final line. For example, two eligible votes for CQRS and one for API composition would produce a CQRS decision with one dissenting view. One vote each and an excluded third proposal would produce an escalation. The actual responses vary.

## Try it

1. After Step 3, set `proposals["Safeguard"]["evidence"] = []`. Run Steps 4–5 and follow Safeguard from the challenge to its revised proposal. Does the revision qualify for a vote?
2. Change the `C1` note in Step 1 to a lookup by order ID across three services. Rerun Steps 1 and 3–5. Which part of `R2` helps explain why API composition may fit?
3. Pick a claim and read its cited note. What extra information would you need to tell whether the claim is well supported?

## Takeaway

Separate proposals let each reviewer start from the same evidence. A chair can ask for a focused critique, apply a vote rule, and leave a record that someone else can inspect.